# Building end-to-end AI Agent in LangChain

- Reference: [CampusX.com](https://colab.research.google.com/drive/1O7cdBtiP_GNXgL9Iz4LPzYfvTKtMtv25?usp=sharing#scrollTo=dISl6fzqZKfU)

In [17]:
!pip install -q --upgrade pip
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    safetensors \
    langchain-huggingface \
    langchain-community \
    requests \
    duckduckgo-search \
    ddgs

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# Import Libraries

In [8]:
import requests
import os

In [2]:
import torch

In [74]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

In [61]:
# from langchain.agents import create_agent
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
# from langchain.agents.structured_output import ToolStrategy

In [31]:
from pydantic import BaseModel

In [42]:
from google.colab import userdata

In [25]:
import langchain

In [26]:
langchain.__version__

'1.1.3'

# Load Model

## Download Model

In [5]:
# Simpler open-source model with no license restrictions
#model_name = "EleutherAI/gpt-neo-125M"  # Smallest, ~500MB
model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for better performance on modern GPUs
)

hf_llm = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

print("Model loaded successfully!")

pipe = pipeline(
    "text-generation",
    model=hf_llm,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.15,
    return_full_text=False,              # 🔥 REQUIRED
    eos_token_id=tokenizer.eos_token_id,
    device_map="auto"  # keep weights offloaded
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded successfully!


In [6]:
llm_model = HuggingFacePipeline(
    pipeline=pipe,
    pipeline_kwargs={
        "stop_sequences": ["<|im_start|>", "<|im_end|>"]  # 🔥 REQUIRED
    }
)

chat_model = ChatHuggingFace(
    llm=llm_model,
    tokenizer=tokenizer,
    chat_template=False                 # 🔥 REQUIRED
)

In [7]:
chat_model.invoke([SystemMessage(content=''), HumanMessage(content="What is the capital of France?")])

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, id='lc_run--019b2c8c-7164-7330-b80c-98c5aa0a9ab4-0')

# Build Tools

In [19]:
search_tool = DuckDuckGoSearchRun()

In [45]:
API_KEY = userdata.get("OPENWEATHER_API_KEY")

@tool
def get_weather(city: str) -> str:
    """
    Take a city name as input and return the current weather in that city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={API_KEY}&units=metric"
    )

    response = requests.get(url).json()

    if response.get("cod") != 200:
        return "City not found"

    temp = response["main"]["temp"]
    desc = response["weather"][0]["description"]

    return f"Temperature: {temp}°C, Condition: {desc}"


In [46]:
get_weather.invoke("London")

'Temperature: 9.59°C, Condition: few clouds'

# Building Agents

In [86]:
react_prompt = PromptTemplate.from_template("""
You are a ReAct agent.

CRITICAL RULES (MUST FOLLOW EXACTLY):
1. You must output ONE and ONLY ONE of the following formats.
2. NEVER output both an Action and a Final Answer in the same response.
3. NEVER simulate or invent tool results.
4. If you output an Action, STOP immediately and output NOTHING ELSE.
5. Use EXACT capitalization: Thought, Action, Action Input, Final Answer.
6. NEVER use lowercase (action:, thought:, etc.).
7. NEVER put Action and Action Input on the same line.

========================
FORMAT A (when a tool is required):

Thought: <your reasoning>
Action: <one of [{tool_names}]>
Action Input: <input for the tool>

========================
FORMAT B (when you are finished):

Final Answer: <your final answer>

========================

TOOLS AVAILABLE:
{tools}

Question:
{input}

========================
Begin.

Thought:
{agent_scratchpad}
""")

In [87]:
tools = [search_tool, get_weather]


# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=chat_model,
    tools=tools,
    prompt=react_prompt
)


# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8
)

In [88]:
# Step 5: Invoke
response = agent_executor.invoke({
    "input": "Find the capital of Bangladesh, then find it's current weather condition"
})

print(response)



> Entering new AgentExecutor chain...
THought: To find the capital of Bangladesh and its current weather, I will first look up the capital using DuckDuckGo Search with the query "capital of Bangladesh", and then look up the current weather of Dhaka, which is the capital of Bangladesh.

Action: duckduckgo_search
Action Input: "capital of Bangladesh"As the capital of the People's Republic of Bangladesh , Dhaka is home to numerous state and diplomatic institutions. The Bangabhaban is the official residence and workplace of the President of Bangladesh , who is the ceremonial head of state under the constitution. Bangladesh Institute of Capital market, not a profit organisation, is a Public Private Institute that has been developed with the acumen of promoting ... ... capital and largest city in Bangladesh ... The city is in Central Bangladesh along the eastern bank of the Buriganga River within the Bengal Delta. Ever since the city was made the capital of Bangladesh , Dhaka has grown in 

In [90]:
response['output']

'Agent stopped due to iteration limit or time limit.'